In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime


## Função Para Ler a Partição

In [0]:
def ler_ultima_particao_delta(spark, base_path):
  """
  Essa fução é para ler a ultima partição dos volumes delta baseada na coluna 'data_processamento'
  """
  try: 
      # Descobrir as partições direto no storage 
      particoes = dbutils.fs.ls(base_path)
      datas = [
              int(p.name.split('=')[1].replace('/', '')) 
              for p in particoes if "data_processamento=" in p.name
      ]
      
      if not datas:
          print(f"Nenhuma partição encontrada em {base_path}")
          return None
      else:
          ultima_particao = max(datas)
          print(f"[{base_path}] Ultima partição: {ultima_particao}")
          return spark.read.format("delta").load(f"{base_path}/data_processamento={ultima_particao}")
  except Exception as e:
    print(f"Erro ao ler caminho {base_path}: {e}")
    return None

## 1. CVM

In [0]:
bronze_path_cvm = "/Volumes/workspace/case_spark_cvm/bronze/cvm_informe_diario/"

df_bronze_cvm = ler_ultima_particao_delta(spark, bronze_path_cvm)

In [0]:
df_bronze_cvm.count()

In [0]:
# Conta a quantidade de valores nulos para cada coluna do dataframe
df_contagem_nulos = df_bronze_cvm.select([
    f.count(f.when(f.col(c).isNull(), c)).alias(c) for c in df_bronze_cvm.columns
])

df_contagem_nulos.show(vertical=True)

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_bronze_cvm = df_bronze_cvm.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.regexp_replace(f.col("CNPJ_FUNDO_CLASSE"), "[^0-9]", "")
)

df_bronze_cvm = df_bronze_cvm.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.col("CNPJ_FUNDO_CLASSE").cast("long").cast("string")
)

#### 1.1.2 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa 
colunas_obrigatorias = ['TP_FUNDO_CLASSE', 'CNPJ_FUNDO_CLASSE', 'VL_TOTAL']

# Aplicando a filtro para dropar as colunas
df_bronze_cvm = df_bronze_cvm.dropna(subset=colunas_obrigatorias)

#### 1.1.3 Retirando dados duplicados

Com a mudança de rosolução da CVM (***Resolução CVM 175***), Com a nova regra, os fundos passaram a ser estruturados em classes e subclasses, adotando o tipo "CLASSES - FIF" (Fundo de Investimento Financeiro).
Caso acha dados do mesmo ***CNPJ_FUNDO_CLASSE***, os dados de "CLASSES - FIF" terão prioridade e o evento com nomecclatura antiga será excluido.

```
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
|TP_FUNDO_CLASSE| CNPJ_FUNDO_CLASSE|ID_SUBCLASSE| DT_COMPTC|   VL_TOTAL|      VL_QUOTA|VL_PATRIM_LIQ|CAPTC_DIA|RESG_DIA|NR_COTST|data_processamento|
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
|  CLASSES - FIF|12.586.174/0001-67|        NULL|2026-01-07|47445220.02|1.406455790000|  47448983.02|     0.00|    0.00|       1|          20260221|
|             FI|12.586.174/0001-67|        NULL|2026-01-07|47445414.65|1.406474270000|  47449606.58|     0.00|    0.00|       1|          20260221|
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
```

In [0]:
# Coluna temporaria para definir prioridade em CLASSES - FIF
df_bronze_cvm = df_bronze_cvm.withColumn(
    "prioridade_tipo",
    f.when(f.col("TP_FUNDO_CLASSE") ==  "CLASSES - FIF", 1).otherwise(2)
)

# Definindo a janela  particionando pelas colunas CORE
window_spec = Window.partitionBy("CNPJ_FUNDO_CLASSE", "ID_SUBCLASSE", "DT_COMPTC").orderBy("prioridade_tipo")

#Aplicamos a numeração das linhas (row_number) dentro de cada janela
df_bronze_cvm = df_bronze_cvm.withColumn("row_num", f.row_number().over(window_spec))

# filtrando prioridade_tipo = 1 de cada grupo e removendo as colunas auxiliares 
df_bronze_cvm = df_bronze_cvm.filter(f.col("row_num") == 1).drop("prioridade_tipo", "row_num")


#### 1.1.4 Tratamento do Tipo de Dado

In [0]:
from datetime import datetime

datetime.fromtimestamp(1061332085)

In [0]:
df_bronze_cvm.filter(f.col("dt_comptc") == '1096927148.94' ).show()

In [0]:
df_bronze_cvm = df_bronze_cvm\
    .withColumn('tp_fundo_classe', f.col('TP_FUNDO_CLASSE').cast(t.StringType()))\
    .withColumn('cnpj_fundo_classe', f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType()))\
    .withColumn('id_subclasse', f.col('ID_SUBCLASSE').cast(t.StringType()))\
    .withColumn('dt_comptc', f.col('DT_COMPTC').cast(t.DateType()))\
    .withColumn('vl_total', f.col('VL_TOTAL').cast(t.DecimalType(38,2)))\
    .withColumn('vl_quota', f.col('VL_QUOTA').cast(t.DecimalType(38,11)))\
    .withColumn('vl_patrim_liq', f.col('VL_PATRIM_LIQ').cast(t.DecimalType(38,2)))\
    .withColumn('captc_dia', f.col('CAPTC_DIA').cast(t.DecimalType(38,2)))\
    .withColumn('resg_dia', f.col('RESG_DIA').cast(t.DecimalType(38,2)))\
    .withColumn('nr_cotst', f.col('NR_COTST').cast(t.LongType()))\
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))

### 1.2 Salvar na camada Silver

In [0]:
display(df_bronze_cvm)

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_bronze_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .format('delta')\
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.case_spark_cvm.silver_cvm_fundos_diario")

## 2. Bank

In [0]:
bronze_path_bank = "/Volumes/workspace/case_spark_cvm/bronze/data_bank/"

df_bronze_bank = ler_ultima_particao_delta(spark, bronze_path_bank)

In [0]:
df_bronze_bank.show()

### 1.1 tratemento silver


#### 1.1.1 Retirando dados nulos de Colunas Cores


In [0]:
# Lista de colunas de caso falte dados precisamos dropa 
colunas_obrigatorias = ['ispb', 'name']

# Aplicando a filtro para dropar as colunas
df_bronze_bank = df_bronze_bank.dropna(subset=colunas_obrigatorias)

#### 1.1.2 Retirando dados duplicados

Caso haja algum problema na origem referente aos dados do ISBP, com essa estratégia conseguimos eliminar os registros duplicados.

In [0]:
#Definindo a janela particionando pelas coluna CORE
window_spec_bank = Window.partitionBy("ispb").orderBy("code")

#Aplicamos a numeração das linhas (row_number) dentro de cada janela
df_bronze_bank = df_bronze_bank.withColumn("row_num", f.row_number().over(window_spec_bank))

df_bronze_bank = df_bronze_bank.filter(f.col("row_num") == 1).drop("row_num") 

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
df_bronze_bank = df_bronze_bank\
    .withColumn('code', f.col('code').cast(t.IntegerType()))\
    .withColumn('fullName', f.col('fullName').cast(t.StringType()))\
    .withColumn('ispb', f.col('ispb').cast(t.StringType()))\
    .withColumn('name', f.col('name').cast(t.StringType()))\
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))

### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_bronze_bank.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_data_bank")

## 3. registro_classe_cvm


In [0]:
bronze_path_registro_classe_cvm = "/Volumes/workspace/case_spark_cvm/bronze/registro_classe_cvm/"

df_registro_classe_cvm = ler_ultima_particao_delta(spark, bronze_path_registro_classe_cvm)

In [0]:
df_registro_classe_cvm.toPandas()

### 1.1 tratemento silver

#### 1.1.1 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa
colunas_obrigatorias = ['ID_Registro_Fundo', 'ID_Registro_Classe', 'CNPJ_Classe', 'Tipo_Classe', 'Data_Inicio','Situacao']

# Aplicando a filtro para dropar as colunas
df_registro_classe_cvm = df_registro_classe_cvm.dropna(subset=colunas_obrigatorias)

#### 1.1.2 Retirando dados duplicados

In [0]:
df_registro_classe_cvm.filter(f.col("CNPJ_Classe") == 32287668000158).toPandas()

Nesta base podem existir registros com ***CNPJ_Classe*** duplicados. Para tratar essa situação, criamos uma coluna de ***prioridade_situacao***, essa coluna irar criar uma regra onde se a situação estiver **"Em Funcionamento Normal"**, vai ter prioridade em seguida mantemos apenas o registro mais recente, considerando o campo ***Data_Registro***.

In [0]:
df_registro_classe_cvm = df_registro_classe_cvm\
    .withColumn("prioridade_situacao", f.when(f.col("Situacao") == "Em Funcionamento Normal", 1).otherwise(2))

window_spec_registros_classe = Window.partitionBy("CNPJ_Classe").orderBy(f.col("prioridade_situacao"), f.col("Data_Registro").desc())

df_registro_classe_cvm = df_registro_classe_cvm.withColumn("row_num", f.row_number().over(window_spec_registros_classe))

df_registro_classe_cvm = df_registro_classe_cvm.filter(f.col("row_num") == 1).drop("row_num", "prioridade_situacao")

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:

df_registro_classe_cvm = df_registro_classe_cvm \
    .withColumn('id_registro_fundo', f.col('ID_Registro_Fundo').cast(t.IntegerType())) \
    .withColumn('id_registro_classe', f.col('ID_Registro_Classe').cast(t.IntegerType())) \
    .withColumn('cnpj_classe', f.col('CNPJ_Classe').cast(t.StringType())) \
    .withColumn('codigo_cvm', f.col('Codigo_CVM').cast(t.IntegerType())) \
    .withColumn('data_registro', f.col('Data_Registro').cast(t.DateType())) \
    .withColumn('data_constituicao', f.col('Data_Constituicao').cast(t.DateType())) \
    .withColumn('data_inicio', f.col('Data_Inicio').cast(t.DateType())) \
    .withColumn('tipo_classe', f.col('Tipo_Classe').cast(t.StringType())) \
    .withColumn('denominacao_social', f.col('Denominacao_Social').cast(t.StringType())) \
    .withColumn('situacao', f.col('Situacao').cast(t.StringType())) \
    .withColumn('data_inicio_situacao', f.col('Data_Inicio_Situacao').cast(t.DateType())) \
    .withColumn('classificacao', f.col('Classificacao').cast(t.StringType())) \
    .withColumn('indicador_desempenho', f.col('Indicador_Desempenho').cast(t.StringType())) \
    .withColumn('classe_cotas', f.col('Classe_Cotas').cast(t.StringType())) \
    .withColumn('classificacao_anbima', f.col('Classificacao_Anbima').cast(t.StringType())) \
    .withColumn('tributacao_longo_prazo', f.col('Tributacao_Longo_Prazo').cast(t.StringType())) \
    .withColumn('entidade_investimento', f.col('Entidade_Investimento').cast(t.StringType())) \
    .withColumn('permitido_aplicacao_cemporcento_exterior', f.col('Permitido_Aplicacao_CemPorCento_Exterior').cast(t.StringType())) \
    .withColumn('classe_esg', f.col('Classe_ESG').cast(t.StringType())) \
    .withColumn('forma_condominio', f.col('Forma_Condominio').cast(t.StringType())) \
    .withColumn('exclusivo', f.col('Exclusivo').cast(t.StringType())) \
    .withColumn('publico_alvo', f.col('Publico_Alvo').cast(t.StringType())) \
    .withColumn('patrimonio_liquido', f.col('Patrimonio_Liquido').cast(t.DecimalType(25,2))) \
    .withColumn('data_patrimonio_liquido', f.col('Data_Patrimonio_Liquido').cast(t.DateType())) \
    .withColumn('cnpj_auditor', f.col('CNPJ_Auditor').cast(t.StringType())) \
    .withColumn('auditor', f.col('Auditor').cast(t.StringType())) \
    .withColumn('cnpj_custodiante', f.col('CNPJ_Custodiante').cast(t.StringType())) \
    .withColumn('custodiante', f.col('Custodiante').cast(t.StringType())) \
    .withColumn('cnpj_controlador', f.col('CNPJ_Controlador').cast(t.StringType())) \
    .withColumn('controlador', f.col('Controlador').cast(t.StringType()))\
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))


In [0]:
display(df_registro_classe_cvm)

### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_registro_classe_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_registro_classe_cvm")

## 4. registro_fundo_cvm

In [0]:
bronze_path_registro_fundo_cvm = "/Volumes/workspace/case_spark_cvm/bronze/registro_fundo_cvm/"

df_registro_fundo_cvm = ler_ultima_particao_delta(spark, bronze_path_registro_fundo_cvm)

In [0]:
df_registro_fundo_cvm.toPandas()

### 1.1 tratemento silver

#### 1.1.1 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa
colunas_obrigatorias = ['ID_Registro_Fundo', 'CNPJ_Fundo', 'Codigo_CVM', 'Tipo_Fundo', 'Situacao']

# Aplicando a filtro para dropar as colunas
df_registro_fundo_cvm = df_registro_fundo_cvm.dropna(subset=colunas_obrigatorias)

#### 1.1.2 Retirando dados duplicados

Nesta base podem existir registros com ***CNPJ_Fundo*** duplicados. Para tratar essa situação, criamos uma coluna de ***prioridade_situacao***, essa coluna irar criar uma regra onde se a situação estiver **"Em Funcionamento Normal"**, vai ter prioridade em seguida mantemos apenas o registro mais recente, considerando o campo ***Data_Registro***.

In [0]:
df_registro_fundo_cvm = df_registro_fundo_cvm\
    .withColumn("prioridade_situacao", f.when(f.col("Situacao") == "Em Funcionamento Normal", 1).otherwise(2))

window_spec_registros_classe = Window.partitionBy("CNPJ_Fundo").orderBy(f.col("prioridade_situacao"), f.col("Data_Registro").desc())

df_registro_fundo_cvm = df_registro_fundo_cvm.withColumn("row_num", f.row_number().over(window_spec_registros_classe))

df_registro_fundo_cvm = df_registro_fundo_cvm.filter(f.col("row_num") == 1).drop("row_num", "prioridade_situacao")

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
df_registro_fundo_cvm = df_registro_fundo_cvm \
    .withColumn('id_registro_fundo', f.col('ID_Registro_Fundo').cast(t.IntegerType())) \
    .withColumn('cnpj_fundo', f.col('CNPJ_Fundo').cast(t.StringType())) \
    .withColumn('codigo_cvm', f.col('Codigo_CVM').cast(t.IntegerType())) \
    .withColumn('data_registro', f.col('Data_Registro').cast(t.DateType())) \
    .withColumn('data_constituicao', f.col('Data_Constituicao').cast(t.DateType())) \
    .withColumn('tipo_fundo', f.col('Tipo_Fundo').cast(t.StringType())) \
    .withColumn('denominacao_social', f.col('Denominacao_Social').cast(t.StringType())) \
    .withColumn('data_cancelamento', f.col('Data_Cancelamento').cast(t.DateType())) \
    .withColumn('situacao', f.col('Situacao').cast(t.StringType())) \
    .withColumn('data_inicio_situacao', f.col('Data_Inicio_Situacao').cast(t.DateType())) \
    .withColumn('data_adaptacao_rcvm175', f.col('Data_Adaptacao_RCVM175').cast(t.DateType())) \
    .withColumn('data_inicio_exercicio_social', f.col('Data_Inicio_Exercicio_Social').cast(t.DateType())) \
    .withColumn('data_fim_exercicio_social', f.col('Data_Fim_Exercicio_Social').cast(t.DateType())) \
    .withColumn('patrimonio_liquido', f.col('Patrimonio_Liquido').cast(t.DecimalType(25,2))) \
    .withColumn('data_patrimonio_liquido', f.col('Data_Patrimonio_Liquido').cast(t.DateType())) \
    .withColumn('diretor', f.col('Diretor').cast(t.StringType())) \
    .withColumn('cnpj_administrador', f.col('CNPJ_Administrador').cast(t.StringType())) \
    .withColumn('administrador', f.col('Administrador').cast(t.StringType())) \
    .withColumn('tipo_pessoa_gestor', f.col('Tipo_Pessoa_Gestor').cast(t.StringType())) \
    .withColumn('cpf_cnpj_gestor', f.col('CPF_CNPJ_Gestor').cast(t.StringType())) \
    .withColumn('gestor', f.col('Gestor').cast(t.StringType())) \
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))


### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_registro_fundo_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_registro_fundo_cvm")

## 5. registro_subclasse_cvm

In [0]:
bronze_path_registro_subclasse_cvm = "/Volumes/workspace/case_spark_cvm/bronze/registro_subclasse_cvm/"

df_registro_subclasse_cvm = ler_ultima_particao_delta(spark, bronze_path_registro_subclasse_cvm)

### 1.1 tratemento silver

#### 1.1.1 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropala 
colunas_obrigatorias = ['ID_Registro_Classe', 'ID_Subclasse', 'Situacao']

# Aplicando a filtro para dropar as colunas
df_registro_subclasse_cvm = df_registro_subclasse_cvm.dropna(subset=colunas_obrigatorias)

#### 1.1.2 Retirando dados duplicados

Em **registro_subclasse** só retiramos os fundos com ***Situacao*** de **Cancelado**

In [0]:
df_registro_subclasse_cvm = df_registro_subclasse_cvm.filter(f.col("Situacao") != 'Cancelado')

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
df_registro_subclasse_cvm = df_registro_subclasse_cvm \
    .withColumn('id_registro_classe', f.col('ID_Registro_Classe').cast(t.IntegerType())) \
    .withColumn('id_subclasse', f.col('ID_Subclasse').cast(t.StringType())) \
    .withColumn('codigo_cvm', f.col('Codigo_CVM').cast(t.IntegerType())) \
    .withColumn('data_constituicao', f.col('Data_Constituicao').cast(t.DateType())) \
    .withColumn('data_inicio', f.col('Data_Inicio').cast(t.DateType())) \
    .withColumn('denominacao_social', f.col('Denominacao_Social').cast(t.StringType())) \
    .withColumn('situacao', f.col('Situacao').cast(t.StringType())) \
    .withColumn('data_inicio_situacao', f.col('Data_Inicio_Situacao').cast(t.DateType())) \
    .withColumn('forma_condominio', f.col('Forma_Condominio').cast(t.StringType())) \
    .withColumn('exclusivo', f.col('Exclusivo').cast(t.StringType())) \
    .withColumn('publico_alvo', f.col('Publico_Alvo').cast(t.StringType())) \
    .withColumn('previdenciario', f.col('Previdenciario').cast(t.StringType())) \
    .withColumn('exclusivo_inr', f.col('Exclusivo_INR').cast(t.StringType())) \
    .withColumn('exclusivo_previdencia_complementar', f.col('Exclusivo_Previdencia_Complementar').cast(t.StringType())) \
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))


### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_registro_subclasse_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_registro_subclasse_cvm")

## 6. Valores Indicador Desempenho

### 1.1 tratemento silver

#### 1.1.1 LEITURA DOS DADOS BRONZE

In [0]:

bronze_path_selic = "/Volumes/workspace/case_spark_cvm/bronze/data_selic/"
df_bronze_selic = ler_ultima_particao_delta(spark, bronze_path_selic)

bronze_path_cdi = "/Volumes/workspace/case_spark_cvm/bronze/data_cdi_diario/"
df_bronze_cdi = ler_ultima_particao_delta(spark, bronze_path_cdi)

bronze_path_ipca = "/Volumes/workspace/case_spark_cvm/bronze/data_ipca_mensal/"
df_bronze_ipca = ler_ultima_particao_delta(spark, bronze_path_ipca)

#### 1.1.2 TRATAMENTO SELIC E CDI

In [0]:
df_selic = df_bronze_selic\
    .withColumn(
    "data",
    f.date_format(f.to_date(f.col("data"), "dd/MM/yyy"), "yyyy-MM-dd")
    )\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("valor", f.col("valor").cast(t.DecimalType(10,2)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("valor", "valor_selic")\


df_cdi = df_bronze_cdi\
    .withColumn(
    "data",
    f.date_format(f.to_date(f.col("data"), "dd/MM/yyy"), "yyyy-MM-dd")
    )\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("valor", f.col("valor").cast(t.DecimalType(10,6)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("valor", "valor_cdi")\




#### 1.1.3 TRATAMENTO IPCA (MENSAL) E CÁLCULO DO ACUMULADO (12 MESES)

In [0]:
df_bronze_ipca = df_bronze_ipca\
    .withColumn("data", f.date_format(f.to_date(f.col("data"), "dd/MM/yyy"), "yyyy-MM-dd"))\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("valor", f.col("valor").cast(t.DecimalType(10,4)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("valor", "ipca_mensal")\
    .withColumnRenamed("data", "data_ipca")
    
# Para calcular o IPCA acumulado de 12 meses corretamente (juros compostos)
# Fator = 1 + (ipca_mensal / 100)
df_ipca = df_bronze_ipca.withColumn("fator", (f.col("ipca_mensal") / 100) + 1)

# Usamos uma Window para pegar os últimos 12 meses ordenados pela data
# Acumulado = (Produto dos fatores de 12 meses) - 1. No PySpark: EXP(SUM(LOG(fator)))
window_12m = Window.orderBy("data_ipca").rowsBetween(-11, Window.currentRow)

df_ipca = df_ipca \
    .withColumn("fator_acumulado", f.exp(f.sum(f.log("fator")).over(window_12m))) \
    .withColumn("ipca_anual", ((f.col("fator_acumulado") - 1) * 100).cast(t.DecimalType(10, 2)))

# Criamos uma chave Ano-Mês para facilitar o join com os dados diários
df_ipca = df_ipca \
    .withColumn("ano_mes", f.date_format("data_ipca", "yyyy-MM")) \
    .select("ano_mes", "ipca_mensal", "ipca_anual")

#### 1.1.4 CRIAÇÃO DE UM CALENDÁRIO ÚNICO E JOIN DOS INDICADORES


In [0]:
# Extraímos todas as datas únicas disponíveis entre Selic e CDI

df_datas = df_selic.select("data").union(df_cdi.select("data")).distinct()

# Criamos a chave Ano-Mês nas datas base
df_datas = df_datas.withColumn("ano_mes", f.date_format("data", "yyyy-MM"))

# Realizamos o Join: left com Selic, left com CDI, left com IPCA
df_indicadores = df_datas \
    .join(df_selic, "data", "left") \
    .join(df_cdi, "data", "left") \
    .join(df_ipca, "ano_mes", "left")




#### 1.1.5 CONTORNO DO PROBLEMA DE IPCA ATRASADO (FORWARD FILL)

In [0]:
# Para os dias cujos meses ainda não têm IPCA lançado (Ex: fev/mar de 2026 ficarão nulos no join),
# preenchemos com o último valor de IPCA conhecido usando a função last() com ignorenulls=True.
window_ffill = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_indicadores = df_indicadores \
    .withColumn("ipca_mensal", f.last("ipca_mensal", ignorenulls=True).over(window_ffill)) \
    .withColumn("ipca_anual", f.last("ipca_anual", ignorenulls=True).over(window_ffill))

df_silver_indicadores = df_indicadores \
    .select("data", "valor_selic", "valor_cdi", "ipca_mensal", "ipca_anual") \
    .withColumn("data_processamento", f.lit(data_proc).cast(t.IntegerType()))

In [0]:
display(df_silver_indicadores.orderBy(f.col("data").desc()))

#### 1.1.5 ESCRITA NA CAMADA SILVER

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_silver_indicadores.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_dados_indicadores_economicos")